In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


### Data Readind

In [0]:
df_cust=spark.read.format("parquet")\
    .load("abfss://bronze@dbenttoend.dfs.core.windows.net/customers")


In [0]:
df_cust.display()

In [0]:
df_cust=df_cust.drop("_rescued_data")
df_cust.display()

In [0]:
df_cust=df_cust.withColumn("domain",split(col("email"), "@").getItem(1))

In [0]:
df_cust.groupBy("domain").agg(count("customer_id").alias("total_cust0")).sort(col("total_cust0").desc()).display()


In [0]:
import time
df_gmail=df_cust.filter(col("domain")=="gmail.com")

df_gmail.display()
time.sleep(5)

df_yahoo=df_cust.filter(col("domain")=="yahoo.com")
df_yahoo.display()
time.sleep(5)

df_hotmail=df_cust.filter(col("domain")=="hotmail.com")
df_hotmail.display()
time.sleep(5)

In [0]:
df_cust=df_cust.withColumn("fullname",concat_ws(" ",col("first_name"),col("last_name")))
df_cust.drop("first_name","last_name")
df_cust.display()


In [0]:
df_cust.write.format("delta").mode("append").save("abfss://silver@dbenttoend.dfs.core.windows.net/customers")

In [0]:
%sql
create schema databricks_catalog.silver

In [0]:
%sql
create table if not exists databricks_catalog.silver.customer_silver
using delta
location 'abfss://silver@dbenttoend.dfs.core.windows.net/customers'

In [0]:
%sql
select * from databricks_catalog.silver.customer_silver